In [1]:
# %%
from pathlib import Path
import sys
import re
import numpy as np
import matplotlib.pyplot as plt
import napari
from scipy.ndimage import binary_opening, label, find_objects

THIS_DIR = Path.cwd()
REPO_ROOT = THIS_DIR.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from Classes.Stitch3D import stitch_volumes


# ============================================================
# SETTINGS
# ============================================================
ROTATION = 0
SHOW_NAPARI = True
MAX_SHIFT = 180

DATA_DIR = Path.cwd().parent / "SYNTHETIC DATA" / "ovlp_xy"

# Input file order:
# True  -> input files are (z, y, x), convert to (z, x, y)
# False -> input files already are (z, x, y)
TRANSPOSE_INPUT_TO_ZXY = True

# Optional features
USE_IGNORE_Z = True
IGNORE_TOP = 15
IGNORE_BOTTOM = 10

USE_CORR_BINARY_MASK = False
CORR_BINARY_THRESHOLD = 0.93

USE_TILED_CORRELATION = True
GRID = (15, 15)

USE_ADAPTIVE_GRID = False
GRID_BINARY_THRESHOLD = CORR_BINARY_THRESHOLD
MIN_VOXELS = 25
TILE_MULTIPLE = (1.5, 1.5)
MIN_GRID = (10, 10)
MAX_GRID = (50, 50)
OPENING_STRUCTURE = np.ones((3, 3, 3), dtype=bool)
SIZE_STATISTIC = "median"

PAIR_SIZE_MODE = "crop"   # "crop" or "pad"
SAVE_OUTPUT = True
OUTPUT_DIR = DATA_DIR / "stitched_output"
OUTPUT_NAME = "stitched_global_volume.npy"

PLOT_PAIR_CORRELATIONS = False
PLOT_TILE_VOTES = False


# ============================================================
# HELPERS
# ============================================================
def validate_threshold(value: float, name: str) -> None:
    if not (0 <= value <= 1):
        raise ValueError(f"{name} must be between 0 and 1")


def get_z_bounds(z_dim: int, ignore_top: int = 0, ignore_bottom: int = 0) -> tuple[int, int]:
    z_start = min(ignore_top, z_dim)
    z_end = max(z_start, z_dim - min(ignore_bottom, z_dim - z_start))
    return z_start, z_end


def make_binary_mask(
    vol: np.ndarray,
    binary_threshold: float | None,
    opening_structure=None,
) -> np.ndarray:
    if binary_threshold is None:
        return np.ones_like(vol, dtype=bool)

    validate_threshold(binary_threshold, "binary_threshold")

    v_abs = np.abs(vol)
    v_max = np.max(v_abs)

    if v_max == 0:
        return np.zeros_like(vol, dtype=bool)

    v_norm = v_abs / v_max
    mask = v_norm >= binary_threshold

    if opening_structure is not None:
        mask = binary_opening(mask, structure=opening_structure)

    return mask


def measure_component_sizes_3d(
    binary_vol: np.ndarray,
    ignore_top: int = 0,
    ignore_bottom: int = 0,
    min_voxels: int = 10,
) -> list[dict]:
    z_dim = binary_vol.shape[0]
    z_start, z_end = get_z_bounds(z_dim, ignore_top=ignore_top, ignore_bottom=ignore_bottom)
    binary_crop = binary_vol[z_start:z_end, :, :]

    labeled, _ = label(binary_crop)
    slices = find_objects(labeled)

    component_sizes = []

    for comp_idx, slc in enumerate(slices, start=1):
        if slc is None:
            continue

        z_slc, x_slc, y_slc = slc
        component_mask = labeled[slc] == comp_idx
        voxels = int(np.count_nonzero(component_mask))

        if voxels < min_voxels:
            continue

        component_sizes.append(
            {
                "label": comp_idx,
                "z_size": int(z_slc.stop - z_slc.start),
                "x_size": int(x_slc.stop - x_slc.start),
                "y_size": int(y_slc.stop - y_slc.start),
                "voxels": voxels,
            }
        )

    return component_sizes


def estimate_adaptive_grid(
    vol1: np.ndarray,
    vol2: np.ndarray,
    axis: str,
    grid_binary_threshold: float,
    ignore_top: int,
    ignore_bottom: int,
    min_voxels: int,
    tile_multiple: tuple[float, float],
    min_grid: tuple[int, int],
    max_grid: tuple[int, int],
    opening_structure,
    size_statistic: str = "median",
) -> tuple[tuple[int, int], dict]:
    mask1 = make_binary_mask(vol1, grid_binary_threshold, opening_structure=opening_structure)
    mask2 = make_binary_mask(vol2, grid_binary_threshold, opening_structure=opening_structure)

    sizes1 = measure_component_sizes_3d(
        mask1,
        ignore_top=ignore_top,
        ignore_bottom=ignore_bottom,
        min_voxels=min_voxels,
    )
    sizes2 = measure_component_sizes_3d(
        mask2,
        ignore_top=ignore_top,
        ignore_bottom=ignore_bottom,
        min_voxels=min_voxels,
    )
    all_sizes = sizes1 + sizes2

    if not all_sizes:
        raise ValueError(
            "No binary components found for adaptive grid sizing. "
            "Try lowering GRID_BINARY_THRESHOLD or MIN_VOXELS."
        )

    z_start, z_end = get_z_bounds(vol1.shape[0], ignore_top=ignore_top, ignore_bottom=ignore_bottom)

    if axis == "x":
        size0_vals = np.array([s["z_size"] for s in all_sizes], dtype=float)
        size1_vals = np.array([s["y_size"] for s in all_sizes], dtype=float)
        plane_dim_0 = z_end - z_start
        plane_dim_1 = vol1.shape[2]
    elif axis == "y":
        size0_vals = np.array([s["z_size"] for s in all_sizes], dtype=float)
        size1_vals = np.array([s["x_size"] for s in all_sizes], dtype=float)
        plane_dim_0 = z_end - z_start
        plane_dim_1 = vol1.shape[1]
    else:
        raise ValueError("axis must be 'x' or 'y'")

    if size_statistic == "mean":
        rep0 = float(np.mean(size0_vals))
        rep1 = float(np.mean(size1_vals))
    else:
        rep0 = float(np.median(size0_vals))
        rep1 = float(np.median(size1_vals))

    tile0 = max(1, int(round(tile_multiple[0] * rep0)))
    tile1 = max(1, int(round(tile_multiple[1] * rep1)))

    raw_rows = max(1, plane_dim_0 // tile0)
    raw_cols = max(1, plane_dim_1 // tile1)

    rows = int(np.clip(raw_rows, min_grid[0], max_grid[0]))
    cols = int(np.clip(raw_cols, min_grid[1], max_grid[1]))

    info = {
        "grid": (rows, cols),
        "raw_grid": (raw_rows, raw_cols),
        "representative_size_axis0": rep0,
        "representative_size_axis1": rep1,
        "tile_size_axis0": tile0,
        "tile_size_axis1": tile1,
        "num_components_vol1": len(sizes1),
        "num_components_vol2": len(sizes2),
        "num_components_total": len(all_sizes),
        "mask1": mask1,
        "mask2": mask2,
    }

    return (rows, cols), info


def normalised_correlation_3d_basic(
    vol1: np.ndarray,
    vol2: np.ndarray,
    axis: str = "x",
    max_shift: int = 100,
    binary_threshold: float | None = None,
    ignore_top: int = 0,
    ignore_bottom: int = 0,
) -> tuple[int, np.ndarray, np.ndarray]:
    z1, x1, y1 = vol1.shape
    z2, x2, y2 = vol2.shape

    if axis not in ("x", "y"):
        raise ValueError("axis must be 'x' or 'y'")

    if binary_threshold is not None:
        validate_threshold(binary_threshold, "binary_threshold")

    z1_start, z1_end = get_z_bounds(z1, ignore_top=ignore_top, ignore_bottom=ignore_bottom)
    z2_start, z2_end = get_z_bounds(z2, ignore_top=ignore_top, ignore_bottom=ignore_bottom)

    vol1 = vol1[z1_start:z1_end]
    vol2 = vol2[z2_start:z2_end]

    z1, x1, y1 = vol1.shape
    z2, x2, y2 = vol2.shape

    v1_abs = np.abs(vol1)
    v2_abs = np.abs(vol2)

    v1_max = np.max(v1_abs)
    v2_max = np.max(v2_abs)

    if binary_threshold is not None:
        mask1_full = np.zeros_like(vol1, dtype=bool) if v1_max == 0 else (v1_abs / v1_max) >= binary_threshold
        mask2_full = np.zeros_like(vol2, dtype=bool) if v2_max == 0 else (v2_abs / v2_max) >= binary_threshold
    else:
        mask1_full = None
        mask2_full = None

    shifts = np.arange(-max_shift, max_shift + 1)
    corr_values = []

    for d in shifts:
        if axis == "x":
            a1_start = max(0, d)
            a1_end = min(x1, x2 + d)

            a2_start = max(0, -d)
            a2_end = min(x2, x1 - d)

            if (a1_end - a1_start) <= 0:
                corr_values.append(0.0)
                continue

            region1 = vol1[:, a1_start:a1_end, :]
            region2 = vol2[:, a2_start:a2_end, :]

            if binary_threshold is not None:
                mask1 = mask1_full[:, a1_start:a1_end, :]
                mask2 = mask2_full[:, a2_start:a2_end, :]
        else:
            a1_start = max(0, d)
            a1_end = min(y1, y2 + d)

            a2_start = max(0, -d)
            a2_end = min(y2, y1 - d)

            if (a1_end - a1_start) <= 0:
                corr_values.append(0.0)
                continue

            region1 = vol1[:, :, a1_start:a1_end]
            region2 = vol2[:, :, a2_start:a2_end]

            if binary_threshold is not None:
                mask1 = mask1_full[:, :, a1_start:a1_end]
                mask2 = mask2_full[:, :, a2_start:a2_end]

        if binary_threshold is not None:
            joint_mask = mask1 & mask2

            if not np.any(joint_mask):
                corr_values.append(0.0)
                continue

            r1 = region1[joint_mask]
            r2 = region2[joint_mask]
        else:
            r1 = region1.ravel()
            r2 = region2.ravel()

        numerator = np.sum(r1 * r2)
        denom = np.sqrt(np.sum(r1 ** 2) * np.sum(r2 ** 2))
        corr_values.append(numerator / denom if denom > 0 else 0.0)

    corr_values = np.array(corr_values, dtype=float)
    best_index = int(np.argmax(corr_values))
    best_shift = int(shifts[best_index])

    return best_shift, shifts, corr_values


def normalised_correlation_3d_tiled(
    vol1: np.ndarray,
    vol2: np.ndarray,
    axis: str = "x",
    max_shift: int = 100,
    grid: tuple[int, int] = (4, 4),
    binary_threshold: float | None = None,
    ignore_top: int = 0,
    ignore_bottom: int = 0,
) -> tuple[int, np.ndarray, np.ndarray, dict]:
    z1, x1, y1 = vol1.shape
    z2, x2, y2 = vol2.shape

    rows, cols = grid
    shifts = np.arange(-max_shift, max_shift + 1)
    corr_values = np.zeros_like(shifts, dtype=float)

    tile_vote_map = np.full((rows, cols), np.nan, dtype=float)
    tile_peak_map = np.full((rows, cols), np.nan, dtype=float)
    valid_tile_count = 0

    z_start, z_end = get_z_bounds(z1, ignore_top=ignore_top, ignore_bottom=ignore_bottom)
    z_usable = z_end - z_start

    if z_usable <= 0:
        raise ValueError("No usable z slices remain after applying ignore_top and ignore_bottom")

    if axis == "x":
        if z1 != z2 or y1 != y2:
            raise ValueError("For x stitching, z and y dimensions must match")

        tile_z = z_usable // rows
        tile_y = y1 // cols

        if tile_z <= 0 or tile_y <= 0:
            raise ValueError(f"Grid {grid} is too fine for volume shape {vol1.shape}")

        for r in range(rows):
            for c in range(cols):
                zs = z_start + r * tile_z
                ze = z_start + (r + 1) * tile_z if r < rows - 1 else z_end

                ys = c * tile_y
                ye = (c + 1) * tile_y if c < cols - 1 else y1

                tile1 = vol1[zs:ze, :, ys:ye]
                tile2 = vol2[zs:ze, :, ys:ye]

                if tile1.size == 0 or tile2.size == 0:
                    continue

                _, _, tile_corr = normalised_correlation_3d_basic(
                    tile1,
                    tile2,
                    axis=axis,
                    max_shift=max_shift,
                    binary_threshold=binary_threshold,
                    ignore_top=0,
                    ignore_bottom=0,
                )

                corr_values += tile_corr
                peak_idx = int(np.argmax(tile_corr))
                tile_vote_map[r, c] = shifts[peak_idx]
                tile_peak_map[r, c] = tile_corr[peak_idx]
                valid_tile_count += 1

    elif axis == "y":
        if z1 != z2 or x1 != x2:
            raise ValueError("For y stitching, z and x dimensions must match")

        tile_z = z_usable // rows
        tile_x = x1 // cols

        if tile_z <= 0 or tile_x <= 0:
            raise ValueError(f"Grid {grid} is too fine for volume shape {vol1.shape}")

        for r in range(rows):
            for c in range(cols):
                zs = z_start + r * tile_z
                ze = z_start + (r + 1) * tile_z if r < rows - 1 else z_end

                xs = c * tile_x
                xe = (c + 1) * tile_x if c < cols - 1 else x1

                tile1 = vol1[zs:ze, xs:xe, :]
                tile2 = vol2[zs:ze, xs:xe, :]

                if tile1.size == 0 or tile2.size == 0:
                    continue

                _, _, tile_corr = normalised_correlation_3d_basic(
                    tile1,
                    tile2,
                    axis=axis,
                    max_shift=max_shift,
                    binary_threshold=binary_threshold,
                    ignore_top=0,
                    ignore_bottom=0,
                )

                corr_values += tile_corr
                peak_idx = int(np.argmax(tile_corr))
                tile_vote_map[r, c] = shifts[peak_idx]
                tile_peak_map[r, c] = tile_corr[peak_idx]
                valid_tile_count += 1

    else:
        raise ValueError("axis must be 'x' or 'y'")

    if valid_tile_count == 0:
        raise ValueError("No valid tiles were found")

    best_index = int(np.argmax(corr_values))
    best_shift = int(shifts[best_index])

    diagnostics = {
        "grid": grid,
        "tile_vote_map": tile_vote_map,
        "tile_peak_map": tile_peak_map,
        "valid_tile_count": valid_tile_count,
    }

    return best_shift, shifts, corr_values, diagnostics


def run_correlation(
    vol1: np.ndarray,
    vol2: np.ndarray,
    *,
    axis: str,
    max_shift: int,
    use_tiled: bool,
    grid: tuple[int, int],
    use_adaptive_grid: bool,
    grid_binary_threshold: float,
    corr_binary_threshold: float | None,
    use_corr_binary_mask: bool,
    use_ignore_z: bool,
    ignore_top: int,
    ignore_bottom: int,
    min_voxels: int,
    tile_multiple: tuple[float, float],
    min_grid: tuple[int, int],
    max_grid: tuple[int, int],
    opening_structure,
    size_statistic: str,
):
    actual_ignore_top = ignore_top if use_ignore_z else 0
    actual_ignore_bottom = ignore_bottom if use_ignore_z else 0
    actual_corr_threshold = corr_binary_threshold if use_corr_binary_mask else None

    diagnostics = {"mode": "simple", "grid_info": None}

    if use_tiled:
        actual_grid = grid

        if use_adaptive_grid:
            actual_grid, grid_info = estimate_adaptive_grid(
                vol1,
                vol2,
                axis=axis,
                grid_binary_threshold=grid_binary_threshold,
                ignore_top=actual_ignore_top,
                ignore_bottom=actual_ignore_bottom,
                min_voxels=min_voxels,
                tile_multiple=tile_multiple,
                min_grid=min_grid,
                max_grid=max_grid,
                opening_structure=opening_structure,
                size_statistic=size_statistic,
            )
            diagnostics["grid_info"] = grid_info

        best_shift, shifts, corr_values, tiled_diag = normalised_correlation_3d_tiled(
            vol1,
            vol2,
            axis=axis,
            max_shift=max_shift,
            grid=actual_grid,
            binary_threshold=actual_corr_threshold,
            ignore_top=actual_ignore_top,
            ignore_bottom=actual_ignore_bottom,
        )
        diagnostics.update(tiled_diag)
        diagnostics["mode"] = "tiled"

    else:
        best_shift, shifts, corr_values = normalised_correlation_3d_basic(
            vol1,
            vol2,
            axis=axis,
            max_shift=max_shift,
            binary_threshold=actual_corr_threshold,
            ignore_top=actual_ignore_top,
            ignore_bottom=actual_ignore_bottom,
        )
        diagnostics["grid"] = None
        diagnostics["tile_vote_map"] = None
        diagnostics["tile_peak_map"] = None
        diagnostics["valid_tile_count"] = None

    return best_shift, shifts, corr_values, diagnostics


# ============================================================
# IO / SHAPE HANDLING
# ============================================================
def parse_tile_indices(path: Path) -> tuple[int, int]:
    match = re.fullmatch(r"volume_ix(\d+)_iy(\d+)\.npy", path.name)
    if match is None:
        raise ValueError(f"Filename does not match expected pattern: {path.name}")
    return int(match.group(1)), int(match.group(2))


def load_volume(path: Path) -> np.ndarray:
    vol_signal = np.load(path).astype(np.float32)
    if TRANSPOSE_INPUT_TO_ZXY:
        return np.transpose(vol_signal, (0, 2, 1))
    return vol_signal


def sort_tile_paths(data_dir: Path) -> dict[int, dict[int, Path]]:
    files = sorted(data_dir.glob("volume_ix*_iy*.npy"))
    if not files:
        raise FileNotFoundError(f"No files matching volume_ix*_iy*.npy found in {data_dir}")

    grid = {}
    for path in files:
        ix, iy = parse_tile_indices(path)
        grid.setdefault(iy, {})[ix] = path
    return grid


def align_pair_for_axis(
    vol1: np.ndarray,
    vol2: np.ndarray,
    axis: str,
    mode: str = "crop",
) -> tuple[np.ndarray, np.ndarray]:
    """
    vol shape assumed (z, x, y)

    axis == 'x': enforce same z and y, allow x to differ
    axis == 'y': enforce same z and x, allow y to differ
    """
    if axis not in ("x", "y"):
        raise ValueError("axis must be 'x' or 'y'")
    if mode not in ("crop", "pad"):
        raise ValueError("mode must be 'crop' or 'pad'")

    z1, x1, y1 = vol1.shape
    z2, x2, y2 = vol2.shape

    if axis == "x":
        target_z = min(z1, z2) if mode == "crop" else max(z1, z2)
        target_y = min(y1, y2) if mode == "crop" else max(y1, y2)

        if mode == "crop":
            vol1_out = vol1[:target_z, :, :target_y]
            vol2_out = vol2[:target_z, :, :target_y]
        else:
            vol1_out = np.zeros((target_z, x1, target_y), dtype=vol1.dtype)
            vol2_out = np.zeros((target_z, x2, target_y), dtype=vol2.dtype)
            vol1_out[:min(z1, target_z), :, :min(y1, target_y)] = vol1[:min(z1, target_z), :, :min(y1, target_y)]
            vol2_out[:min(z2, target_z), :, :min(y2, target_y)] = vol2[:min(z2, target_z), :, :min(y2, target_y)]

    else:
        target_z = min(z1, z2) if mode == "crop" else max(z1, z2)
        target_x = min(x1, x2) if mode == "crop" else max(x1, x2)

        if mode == "crop":
            vol1_out = vol1[:target_z, :target_x, :]
            vol2_out = vol2[:target_z, :target_x, :]
        else:
            vol1_out = np.zeros((target_z, target_x, y1), dtype=vol1.dtype)
            vol2_out = np.zeros((target_z, target_x, y2), dtype=vol2.dtype)
            vol1_out[:min(z1, target_z), :min(x1, target_x), :] = vol1[:min(z1, target_z), :min(x1, target_x), :]
            vol2_out[:min(z2, target_z), :min(x2, target_x), :] = vol2[:min(z2, target_z), :min(x2, target_x), :]

    return vol1_out, vol2_out


# ============================================================
# PAIRWISE ORIGINAL-TILE STITCHES
# ============================================================
def measure_pair_shift(
    vol_a: np.ndarray,
    vol_b: np.ndarray,
    axis: str,
    label_a: str,
    label_b: str,
) -> dict:
    vol_a_aligned, vol_b_aligned = align_pair_for_axis(
        vol_a,
        vol_b,
        axis=axis,
        mode=PAIR_SIZE_MODE,
    )

    best_shift, shifts, corr_values, diagnostics = run_correlation(
        vol_a_aligned,
        vol_b_aligned,
        axis=axis,
        max_shift=MAX_SHIFT,
        use_tiled=USE_TILED_CORRELATION,
        grid=GRID,
        use_adaptive_grid=USE_ADAPTIVE_GRID,
        grid_binary_threshold=GRID_BINARY_THRESHOLD,
        corr_binary_threshold=CORR_BINARY_THRESHOLD,
        use_corr_binary_mask=USE_CORR_BINARY_MASK,
        use_ignore_z=USE_IGNORE_Z,
        ignore_top=IGNORE_TOP,
        ignore_bottom=IGNORE_BOTTOM,
        min_voxels=MIN_VOXELS,
        tile_multiple=TILE_MULTIPLE,
        min_grid=MIN_GRID,
        max_grid=MAX_GRID,
        opening_structure=OPENING_STRUCTURE,
        size_statistic=SIZE_STATISTIC,
    )

    print(f"[{axis}] {label_a} -> {label_b} shift={best_shift}")
    print(f"    original: {vol_a.shape} vs {vol_b.shape}")
    print(f"    aligned : {vol_a_aligned.shape} vs {vol_b_aligned.shape}")

    if diagnostics["grid"] is not None:
        print(f"    grid={diagnostics['grid']}, valid_tiles={diagnostics['valid_tile_count']}")

    if PLOT_PAIR_CORRELATIONS:
        plt.figure(figsize=(8, 4.5))
        plt.plot(shifts, corr_values, linewidth=1.8)
        plt.title(f"{axis}-stitch: {label_a} -> {label_b}")
        plt.xlabel("Pixel Shift")
        plt.ylabel("Correlation")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    if PLOT_TILE_VOTES and diagnostics["tile_vote_map"] is not None:
        plt.figure(figsize=(6, 5))
        heat = np.ma.masked_invalid(diagnostics["tile_vote_map"])
        im = plt.imshow(heat, aspect="auto", interpolation="nearest")
        plt.colorbar(im, label="Tile best shift")
        plt.title(f"Tile Vote Map: {label_a} -> {label_b}")
        plt.xlabel("Tile column")
        plt.ylabel("Tile row")
        plt.tight_layout()
        plt.show()

    return {
        "axis": axis,
        "label_a": label_a,
        "label_b": label_b,
        "best_shift": int(best_shift),
        "input_shape_a": tuple(vol_a.shape),
        "input_shape_b": tuple(vol_b.shape),
        "aligned_shape_a": tuple(vol_a_aligned.shape),
        "aligned_shape_b": tuple(vol_b_aligned.shape),
        "shifts": shifts,
        "corr_values": corr_values,
        "diagnostics": diagnostics,
    }


def compute_pairwise_shifts(tile_grid: dict[int, dict[int, Path]]):
    """
    Computes:
    - x shifts for horizontal neighbors within each row
    - exactly one y shift per row transition, using the first tile in the new row

    Example:
      row 0: x shifts across row
      row 0 -> row 1: one y shift using first tile in row 1
      row 1: x shifts across row
      row 1 -> row 2: one y shift using first tile in row 2
    """
    y_values = sorted(tile_grid.keys())

    loaded = {}
    for iy in y_values:
        for ix in sorted(tile_grid[iy].keys()):
            loaded[(ix, iy)] = load_volume(tile_grid[iy][ix])

    x_shifts = {}
    row_anchor_y_shifts = {}
    diagnostics = []

    for row_idx, iy in enumerate(y_values):
        row_x = sorted(tile_grid[iy].keys())

        # Build this row using only x-neighbor stitches
        for i in range(len(row_x) - 1):
            ix0 = row_x[i]
            ix1 = row_x[i + 1]

            vol_a = loaded[(ix0, iy)]
            vol_b = loaded[(ix1, iy)]

            info = measure_pair_shift(
                vol_a,
                vol_b,
                axis="x",
                label_a=f"ix{ix0:02d}_iy{iy:02d}",
                label_b=f"ix{ix1:02d}_iy{iy:02d}",
            )
            x_shifts[(ix0, iy, ix1, iy)] = info["best_shift"]
            diagnostics.append(info)

        # Anchor the next row using exactly one y-stitch
        if row_idx < len(y_values) - 1:
            iy_next = y_values[row_idx + 1]

            anchor_ix = sorted(tile_grid[iy_next].keys())[0]
            if anchor_ix not in tile_grid[iy]:
                raise ValueError(
                    f"Cannot anchor row iy={iy_next} from row iy={iy}: "
                    f"anchor ix={anchor_ix} missing in previous row"
                )

            vol_a = loaded[(anchor_ix, iy)]
            vol_b = loaded[(anchor_ix, iy_next)]

            info = measure_pair_shift(
                vol_a,
                vol_b,
                axis="y",
                label_a=f"ix{anchor_ix:02d}_iy{iy:02d}",
                label_b=f"ix{anchor_ix:02d}_iy{iy_next:02d}",
            )
            row_anchor_y_shifts[(iy, iy_next)] = {
                "anchor_ix": anchor_ix,
                "shift_y": info["best_shift"],
            }
            diagnostics.append(info)

    return loaded, x_shifts, row_anchor_y_shifts, diagnostics

# ============================================================
# GLOBAL POSITIONS
# ============================================================
def accumulate_positions(tile_grid, x_shifts, row_anchor_y_shifts):
    """
    positions[(ix, iy)] = (x_offset, y_offset)

    Strategy:
    - first row built entirely from x shifts
    - each next row gets exactly one y-anchor shift
    - then that row is built across in x only
    """
    y_values = sorted(tile_grid.keys())
    positions = {}

    first_iy = y_values[0]
    first_row_x = sorted(tile_grid[first_iy].keys())
    first_ix = first_row_x[0]
    positions[(first_ix, first_iy)] = (0, 0)

    # Build first row from x shifts
    for i in range(1, len(first_row_x)):
        ix_prev = first_row_x[i - 1]
        ix_curr = first_row_x[i]
        prev_x, prev_y = positions[(ix_prev, first_iy)]
        shift_x = x_shifts[(ix_prev, first_iy, ix_curr, first_iy)]
        positions[(ix_curr, first_iy)] = (prev_x + shift_x, prev_y)

    # Build remaining rows:
    # one y anchor, then x only
    for row_idx in range(1, len(y_values)):
        iy_prev = y_values[row_idx - 1]
        iy_curr = y_values[row_idx]

        anchor_info = row_anchor_y_shifts[(iy_prev, iy_curr)]
        anchor_ix = anchor_info["anchor_ix"]
        shift_y = anchor_info["shift_y"]

        base_x, base_y = positions[(anchor_ix, iy_prev)]
        positions[(anchor_ix, iy_curr)] = (base_x, base_y + shift_y)

        row_x = sorted(tile_grid[iy_curr].keys())

        # Fill left from anchor if needed
        anchor_pos = row_x.index(anchor_ix)
        for i in range(anchor_pos - 1, -1, -1):
            ix_left = row_x[i]
            ix_right = row_x[i + 1]

            right_x, right_y = positions[(ix_right, iy_curr)]
            shift_x = x_shifts[(ix_left, iy_curr, ix_right, iy_curr)]
            positions[(ix_left, iy_curr)] = (right_x - shift_x, right_y)

        # Fill right from anchor
        for i in range(anchor_pos + 1, len(row_x)):
            ix_prev = row_x[i - 1]
            ix_curr2 = row_x[i]

            prev_x, prev_y = positions[(ix_prev, iy_curr)]
            shift_x = x_shifts[(ix_prev, iy_curr, ix_curr2, iy_curr)]
            positions[(ix_curr2, iy_curr)] = (prev_x + shift_x, prev_y)

    return positions


# ============================================================
# GLOBAL CANVAS
# ============================================================
def place_tiles_on_canvas(loaded: dict, positions: dict) -> tuple[np.ndarray, dict]:
    z_max = max(vol.shape[0] for vol in loaded.values())

    min_x = min(pos[0] for pos in positions.values())
    min_y = min(pos[1] for pos in positions.values())

    shifted_positions = {
        key: (pos[0] - min_x, pos[1] - min_y)
        for key, pos in positions.items()
    }

    max_x = 0
    max_y = 0
    for key, vol in loaded.items():
        x0, y0 = shifted_positions[key]
        _, x_size, y_size = vol.shape
        max_x = max(max_x, x0 + x_size)
        max_y = max(max_y, y0 + y_size)

    canvas = np.zeros((z_max, max_x, max_y), dtype=np.float32)

    for key, vol in loaded.items():
        x0, y0 = shifted_positions[key]
        z_size, x_size, y_size = vol.shape

        existing = canvas[:z_size, x0:x0 + x_size, y0:y0 + y_size]
        canvas[:z_size, x0:x0 + x_size, y0:y0 + y_size] = np.maximum(existing, vol)

    return canvas, shifted_positions


# ============================================================
# MAIN
# ============================================================
tile_grid = sort_tile_paths(DATA_DIR)

print(f"Found {sum(len(v) for v in tile_grid.values())} tiles in {DATA_DIR}")
print("Tiles:")
for iy in sorted(tile_grid.keys()):
    for ix in sorted(tile_grid[iy].keys()):
        print(f"  ix{ix:02d}_iy{iy:02d}")

loaded, x_shifts, row_anchor_y_shifts, all_diagnostics = compute_pairwise_shifts(tile_grid)
positions = accumulate_positions(tile_grid, x_shifts, row_anchor_y_shifts)
full_volume, shifted_positions = place_tiles_on_canvas(loaded, positions)

print("\nTile positions:")
for key in sorted(shifted_positions):
    print(f"  {key}: {shifted_positions[key]}")

print("\nFinal stitched volume shape (z, x, y):", full_volume.shape)


# ============================================================
# SAVE
# ============================================================
if SAVE_OUTPUT:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    out_path = OUTPUT_DIR / OUTPUT_NAME
    np.save(out_path, full_volume)
    print(f"Saved stitched volume to: {out_path}")


# ============================================================
# OPTIONAL NAPARI VIEW
# ============================================================
if SHOW_NAPARI:
    viewer = napari.Viewer()

    # ----------------------------------------
    # Original tiles
    # ----------------------------------------
    for key in sorted(loaded):
        ix, iy = key
        vol = loaded[key]

        viewer.add_image(
            np.transpose(vol, (0, 2, 1)),
            name=f"orig_ix{ix:02d}_iy{iy:02d}",
            visible=False,
        )

    # ----------------------------------------
    # Placed tiles in global stitched coordinates
    # ----------------------------------------
    for key in sorted(shifted_positions):
        ix, iy = key
        vol = loaded[key]
        x0, y0 = shifted_positions[key]

        z_size, x_size, y_size = vol.shape

        placed_canvas = np.zeros_like(full_volume, dtype=np.float32)
        placed_canvas[:z_size, x0:x0 + x_size, y0:y0 + y_size] = vol

        viewer.add_image(
            np.transpose(placed_canvas, (0, 2, 1)),
            name=f"placed_ix{ix:02d}_iy{iy:02d}",
            visible=True,
            opacity=0.6,
        )

    # ----------------------------------------
    # Final stitched volume
    # ----------------------------------------
    viewer.add_image(
        np.transpose(full_volume, (0, 2, 1)),
        name="stitched_full_volume",
        colormap="inferno",
        visible=True,
    )

    viewer.dims.axis_labels = ("z", "y", "x")
    napari.run()

Found 9 tiles in /Users/ottobruce-gardyne/Documents/Year4/GIP/signal-processing-G2066/SYNTHETIC DATA/ovlp_xy
Tiles:
  ix00_iy00
  ix01_iy00
  ix02_iy00
  ix00_iy01
  ix01_iy01
  ix02_iy01
  ix00_iy02
  ix01_iy02
  ix02_iy02
[x] ix00_iy00 -> ix01_iy00 shift=-59
    original: (400, 200, 200) vs (400, 200, 200)
    aligned : (400, 200, 200) vs (400, 200, 200)
    grid=(15, 15), valid_tiles=225
[x] ix01_iy00 -> ix02_iy00 shift=-60
    original: (400, 200, 200) vs (400, 200, 200)
    aligned : (400, 200, 200) vs (400, 200, 200)
    grid=(15, 15), valid_tiles=225
[y] ix00_iy00 -> ix00_iy01 shift=-58
    original: (400, 200, 200) vs (400, 200, 200)
    aligned : (400, 200, 200) vs (400, 200, 200)
    grid=(15, 15), valid_tiles=225
[x] ix00_iy01 -> ix01_iy01 shift=-60
    original: (400, 200, 200) vs (400, 200, 200)
    aligned : (400, 200, 200) vs (400, 200, 200)
    grid=(15, 15), valid_tiles=225
[x] ix01_iy01 -> ix02_iy01 shift=-59
    original: (400, 200, 200) vs (400, 200, 200)
    aligne